# 从 Attention 到 TTT：线性化之路（3/5）—— 流式更新：参数 W 成为 RNN 状态

第 2 册把输出写成 $o = \sum a_i \mu_W(k_i)$，其中 $\mu_W$ 是个待训练的预测模型。
这一册回答：**$W$ 从哪来？** 文章的做法是把历史样本依次喂进来，用 SGD 边走边更新。

## 1. 训练目标：最大似然 = 最小化平方误差

把历史 $(k_1, v_1), \dots, (k_n, v_n)$ 看成依次到达的训练数据。
概率模型最自然的目标是最大似然，也就是最小化负对数似然。
对正态分布而言，负对数似然正比于预测残差的平方：

$$-\log p(v_i \mid k_i) \;\propto\; \tfrac{1}{2}\Vert v_i - \mu_W(k_i)\Vert^2 \;\triangleq\; \ell(W_{t-1}; k_t, v_t)$$

直觉：模型拿 $k_t$ 去预测 $v_t$，预测得越准，损失越小。

## 2. 一步梯度下降，就是一条递推式

每来一个新样本 $(k_t, v_t)$，执行一步梯度下降：

$$W_t = W_{t-1} - \eta_t \nabla_{W_{t-1}} \, \ell(W_{t-1}; k_t, v_t)$$

- $W_{t-1}$：上一步的参数；$W_t$：更新后的参数；
- $\eta_t$：学习率；
- 整个过程里，**$W$ 是唯一的状态变量**，它基于当前输入不断修正自己。

这在结构上就是一个 **RNN**：输入是 $(k_t, v_t)$，状态是 $W$，每步按递推式更新。
它把"V 相对于 K 的增量信息"压缩进了有限维的参数 $W$ 里。

In [ ]:
import numpy as np

rng = np.random.default_rng(2)
d = 3
T = 12  # 依次到达 12 个样本

# 造一个隐式线性关系 v = W_true k + 噪声，用来观察预测误差下降
W_true = rng.standard_normal((d, d))
K = rng.standard_normal((T, d))
K /= np.linalg.norm(K, axis=1, keepdims=True)   # 归一化, 让梯度稳定
V = (K @ W_true.T) + 0.1 * rng.standard_normal((T, d))

# 预测模型 μ(k) = W k，W 随机初始化
W = rng.standard_normal((d, d)) * 0.1
eta = 0.03

errors = []
for t in range(T):
    k, v = K[t], V[t]
    pred = W @ k
    residual = pred - v                      # 预测 - 真实
    loss = 0.5 * np.sum(residual ** 2)
    errors.append(loss)
    # 梯度: ℓ = ½‖Wk - v‖²  →  ∂ℓ/∂W = (Wk - v) kᵀ
    grad = np.outer(residual, k)
    W = W - eta * grad                        # 一步 SGD，得到 W_t

print("每步损失:", np.round(errors, 3))
print("最后一步损失: %.4f" % errors[-1])
print()
print("观察: 损失总体下降 —— W 在边读边把 K→V 的规律压缩进自己的参数里")

## 3. 和 TTT 是什么关系

这个递归形式和 **TTT（Test-Time Training，测试时训练）** 实际上是一回事：

- 普通 RNN：状态是固定维度的隐向量；
- 这里：状态 $W$ 本身就是一个"边读边训练的小模型"，读到新内容就把它训进去；
- 区别只在出发点：本文是为了给 KV Cache 减负，压缩给定 K 之后的 V 信息；
  TTT 则直接定义了完整的写入和读出过程。

记住这条线索：**状态 $W$ + 流式更新 = TTT 雏形**，第 4 册会补上读出。

## 4. 小结（3/5）

- 训练目标是最大似然，正态分布下等价于最小化预测残差平方；
- 一步 SGD 写成递推式 $W_t = W_{t-1} - \eta \nabla \ell$，$W$ 成为 RNN 状态；
- 样本依次到达、参数边读边更新，就是 TTT 的递归形式；
- 到此 V 已经被压进 $W$，但 **K 还在**，式 (4) 仍是 softmax 形式，复杂度还没降。

→ **下一册**：两次"平均"近似，先把 $\mu$ 搬进求和号里，再把输入换成 $q$，
得到干净的读出 $o = \mu_W(q)$。